## Generative models for modality integration

In [ ]:
import os
import sys
import gc

import tifffile
import numpy as np
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from ml_collections import ConfigDict
from skimage import io
from skimage.transform import rescale
from skimage.exposure import rescale_intensity
from skimage.color import rgb2gray

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util/')

import registration, IO, utils, zonation, plot

from importlib import reload
reload(plot)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

### 3D integration of DESI (pos & neg ions) + Post DESI modalities

- Load 3D aligned DESI + post-DESI H&E & CyIF images; select ROI patches for training
- Resize high-res H&E & CyIF ($\sim 2\mu m$ towards DESI ($\sim 40\mu m$)
- Create noisy VGAE structural priors $t_i \text{ s.t. } p(u_i) = \mathcal{N}(t_i, \sigma^2 I)$ with heat diffusion from H&E and CyIF

Load multi-modal aligned images:

In [ ]:
integrated_path = '../data/integrated/'

he_path = '../data/desi/post_desi_HE_aligned/'
he_filenames = [os.path.join(he_path, f) for f in sorted(os.listdir(he_path)) if f[-3:] == 'tif']
he_aligned = [tifffile.imread(f) for f in he_filenames]

cyif_path = '../data/desi/post_desi_cycif_aligned/'
cyif_filenames = [os.path.join(cyif_path, f) for f in sorted(os.listdir(cyif_path)) if f[-3:] == 'tif']
cyif_aligned = [tifffile.imread(f) for f in cyif_filenames]

desi_path = '../data/desi/desi_aligned/'
desi_filenames = [os.path.join(desi_path, f) for f in sorted(os.listdir(desi_path)) if f[-3:] == 'tif']
desi_aligned = [tifffile.imread(f) for f in desi_filenames]

In [ ]:
def trim_fov(img: np.ndarray,
             ylim: tuple = None, xlim: tuple = None,
             radius: int = None , channel_axis: int = 0):
    """Create trimmed FOV image stacks"""
    assert img.ndim == 3, "Requires multi-channel input image"
    assert channel_axis == 0 or channel_axis == 2, \
        "Requires dimension ordering as [C, Y, X] or [Y, X, C]"
    raw = img.copy() if channel_axis == 0 else img.transpose(2,0,1)
    ny, nx = raw.shape[-2:]
    if isinstance(ylim, tuple) and isinstance(xlim, tuple):
        assert 0 <= ylim[0] < ylim[1] < ny and 0 <= xlim[0] < xlim[1] < nx, \
            "Invalid trimming ROI range"
        ylow, yhigh = ylim
        xlow, xhigh = xlim
    else:
        radius = min(radius, ny//2, nx//2)
        ylow, yhigh = ny//2 - radius, ny//2 + radius
        xlow, xhigh = nx//2 - radius, nx//2 + radius
    trimmed = raw[:, ylow:yhigh, xlow:xhigh] if channel_axis == 0 \
              else raw[:, ylow:yhigh, xlow:xhigh].transpose(1,2,0)
    return trimmed


Get trimmed FOV for DESI:

In [ ]:
# Load DESI ion labels
ifile = open(desi_filenames[0], 'rb')
ion_labels = IO.load_ome_labels(ifile, desi_filenames[0])
ifile.close()
del ifile


# Save DESI FOVs
desi_outdir = os.path.join(integrated_path, 'desi')
if not os.path.exists(desi_outdir):
    os.makedirs(desi_outdir, exist_ok=True)
    
desi_trimmed = [trim_fov(img, radius=128) for img in desi_aligned]
annot_desi_trimmed = {
    'DESI_slice_' + filename.split('_')[-1]: {
        label: chan
        for (label, chan) in zip(ion_labels, img)
    }
    for (filename, img) in zip(desi_filenames, desi_trimmed)
}

IO.save_annot_tiffs(annot_desi_trimmed, path=desi_outdir)
gc.collect()

Get corresponding trimmed FOV for H&E + CyCIF:

In [ ]:
# Save trimmed CyIF & H&E FOVs
ratio = np.round(cyif_aligned[0].shape[-1] / desi_aligned[0].shape[-1]).astype(np.uint8)
cyif_trimmed = [trim_fov(img, radius=128*ratio) for img in cyif_aligned]

cyif_outdir = os.path.join(integrated_path, 'cycif')
if not os.path.exists(cyif_outdir):
    os.makedirs(cyif_outdir, exist_ok=True)
for filename, img in zip(cyif_filenames, cyif_trimmed):
    tifffile.imwrite(os.path.join(cyif_outdir, filename.split('/')[-1]), img)

he_trimed = [trim_fov(img, radius=128*ratio, channel_axis=2) for img in he_aligned]

he_outdir = os.path.join(integrated_path, 'he')
if not os.path.exists(he_outdir):
    os.makedirs(he_outdir, exist_ok=True)
for filename, img in zip(he_filenames, he_trimed):
    tifffile.imwrite(os.path.join(he_outdir, filename.split('/')[-1]), img)

gc.collect()

Generate Low-res Heat diffusion priors from CyIF images

In [ ]:
# Load CyIF & H&E FOVs
integrated_path = '../data/integrated/'
cyif_imgs = [utils.norm_by_channel(tifffile.imread(os.path.join(integrated_path, 'cycif', f)))
             for f in sorted(os.listdir(integrated_path))
             if f[-3:] == 'tif' and 'CycIF' in f]

desi_imgs = [tifffile.imread(os.path.join(integrated_path, 'desi', f))
           for f in sorted(os.listdir(integrated_path))
           if f[-3:] == 'tif' and 'DESI' in f]


# Downscale cyif images to DESI dimension
# Per-channel value range: [0, 1]
cyif_ny, cyif_nx = cyif_imgs[0].shape[1:]
desi_ny, desi_nx = desi_imgs[0].shape[1:]
assert cyif_ny // desi_ny == cyif_nx // desi_nx, "Ratio along x & y dimension across modalities should be the same"

ratio = desi_ny / cyif_ny
cyif_imgs_ds = [utils.norm_by_channel(rescale(img, scale=ratio, channel_axis=0)) 
                for img in cyif_imgs]

# Select zonation relevant channels: [GS, CYP, ASS]
cyif_zonations = [img[[2,6,1]] for img in cyif_imgs_ds]

del cyif_imgs, cyif_imgs_ds
gc.collect()

In [ ]:
# Generate src (CV) & sink (PV) priors from CyCIF images
cyif_vein_masks = [utils.create_vein_mask(src_chan=img[0],         # CV+ 
                                          sink_chan=1-img.max(0),  # (CV- & CYP- & ASS-)
                                          q=0.05)
                   for img in cyif_zonations]
# tifffile.imwrite('../data/desi/cyif_priors_stacked.tif', np.array(cyif_vein_masks)) 

desi_filenames = [f.split('.')[0] for f in sorted(os.listdir('../data/desi/desi_aligned/'))
                  if f[-3:] == 'tif' and 'DESI' in f]

u_prior_path = os.path.join(integrated_path, 'zonation_prior')
if not os.path.exists(u_prior_path):
    os.makedirs(u_prior_path, exist_ok=True)

u_priors = []
for filename, vein_mask in zip(desi_filenames, cyif_vein_masks):
    hd_model = zonation.HeatDiffusion(vein_prior=vein_mask, ndim=2)
    _, _ = hd_model.get_interior_U()
    u_prior = hd_model.infer_zone_dynamics()
    u_priors.append(u_prior)
    print('Saving zonation heat diffusion priors for {}...'.format(filename))
    tifffile.imwrite(os.path.join(u_prior_path, filename + '_prior.tif'), u_prior)

In [ ]:
plot.disp_chans(cyif_vein_masks, cmap='coolwarm', title='Zonation Vein masks')
plot.disp_chans(u_priors, cmap='coolwarm', title=r'Zonation priors $t$')

---

### (Archived): experiments w/ heat diffusion generation

**DEBUG: equal balanced pixel sampling for CV / PV priors**

In [ ]:
# Sample equal # CV / PV boundary pts
def sample_priors(prior, is_3d=False, n=2000):
    cv_coords = np.where(prior == 1)
    indices = np.random.choice(np.arange(len(cv_coords[0])), size=n, replace=False)
    if is_3d:
        sampled_cv_coords = tuple([cv_coords[0][indices], 
                                   cv_coords[1][indices],
                                   cv_coords[2][indices]])
    else:
        sampled_cv_coords = tuple([cv_coords[0][indices], 
                                   cv_coords[1][indices]])

    pv_coords = np.where(prior == -1)
    indices = np.random.choice(np.arange(len(pv_coords[0])), size=n, replace=False)
    if is_3d:
        sampled_pv_coords = tuple([pv_coords[0][indices], 
                                   pv_coords[1][indices],
                                   pv_coords[2][indices]])
    else:
        sampled_pv_coords = tuple([pv_coords[0][indices], 
                                   pv_coords[1][indices]])

    masked_prior = np.zeros_like(prior)
    masked_prior[sampled_cv_coords] = 1
    masked_prior[sampled_pv_coords] = -1

    return masked_prior

masked_vein_priors = np.array([sample_priors(img)
                               for img in vein_priors])

In [ ]:
masked_vein_priors = rescale(masked_vein_priors, 0.125, preserve_range=True, channel_axis=0)

In [ ]:
# 2D heat diffusion
# Us_2d = []  # Heat along axial-direction
# for vein_prior in vein_priors:
#     model = zonation.HeatDiffusion(vein_prior=vein_prior, ndim=2)
#     U_i, interior_nodes = model.get_interior_U()
#     U = model.infer_zone_dynamics()
#     Us_2d.append(U)


# 3D heat diffusion 
model = zonation.HeatDiffusion(vein_prior=np.array(masked_vein_priors), ndim=3, anis=20)
U_i, interior_nodes = model.get_interior_U()
Us = model.infer_zone_dynamics()
lobule_layers = model.infer_zones()


In [ ]:
def disp_zone_temps(imgs, title=None, ncols=4, cmap='coolwarm'):
    """Display 3-channel zonation image"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx], cmap=cmap)
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
disp_zone_temps(lobule_layers, title='Zonations', cmap='plasma')

### (Archived): $\beta$-VAE for Cy-IF reconstruction

- Benchmark performance w/ linear layers vs. Conv2d
- Uninformative priors $p(z)$ (trying full covariance other than diag.)

In [ ]:
from importlib import reload

reload(utils)
reload(configs)
reload(bvae)
reload(model_train)
reload(model_eval)

#### Training b-VAE 1D

In [ ]:
model_configs = configs.set_model_configs(
    drop_rate = 0.2,
    beta = 0.33,
    pz_std = 1,
    c_base = 128,
    layer_mults = [8, 4],
    ydim = 128, 
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

train_configs = configs.set_train_configs(
    data_path = '../data/cycif/zonation_lowres/',
    lr = 1e-4,
    n_epochs = 100
)

dataloader = DataLoader(
    IMSDataset(
        data_path=train_configs.data_path,
        norm_stats=(constants.CYIF_MEAN, constants.CYIF_STD)
    ),
    batch_size=model_configs.batch_size
)


In [ ]:
model = bvae.BetaVAE(model_configs)

model, loss_dict = model_train.train(model,
                                     dataloader,
                                     train_configs, 
                                     model_configs)

In [ ]:
loss_tot = torch.tensor(loss_dict['total']).cpu().numpy()
loss_nll = torch.tensor(loss_dict['NLL']).cpu().numpy()
loss_kl = torch.tensor(loss_dict['KL']).cpu().numpy()

In [ ]:
n_epochs = train_configs.n_epochs
beta = model_configs.beta

step = 10

plt.figure(figsize=(6, 4))
plt.plot(np.arange(n_epochs)[::step], loss_tot[::step], '.--',  label = 'Total loss')
plt.plot(np.arange(n_epochs)[::step], loss_nll[::step], '.--', color='orange', label = r'$\Vert x - \hat{x} \Vert^2$')
plt.plot(np.arange(n_epochs)[::step], loss_kl[::step], '.--', color='green', label = r'$D_{\text{KL}}(q(z \mid x) || p(z))$')

plt.title('Training logs')
plt.legend()
plt.show()

In [ ]:
torch.save(model.state_dict(), 'results/bvae_1d_lowres_std_norm.pt')

#### Training b-VAE 2D

- **Delete L indices 0, 1, 2, 16, 18**

In [ ]:
# Model configs
model_configs = configs.set_model_configs(
    c_in = 8,
    c_out = 8,
    drop_rate = 0.1,
    beta = 0.5,
    pz_std = 1,
    ydim = 512,
    xdim = 512,
    latent_dim = 64,
    device = torch.device('cpu')
)

train_configs = configs.set_train_configs(
    data_path = '../data/cycif/zonation_hires/',
    prior_path = '../data/cycif/zonation_mask/',
    lr = 1e-4,
    n_epochs = 100
)

dataloader = DataLoader(
    IMSDataset(
        data_path=train_configs.data_path,
        prior_path=train_configs.prior_path,
        norm_stats=(constants.CYIF_MEAN, constants.CYIF_STD)
    ),
    batch_size=model_configs.batch_size
)

In [ ]:
model = bvae.BetaVAE2D(model_configs)

model, loss_dict = model_train.train(model,
                                     dataloader,
                                     train_configs, 
                                     model_configs)

In [ ]:
loss_tot = torch.tensor(loss_dict['total']).cpu().numpy()
loss_nll = torch.tensor(loss_dict['NLL']).cpu().numpy()
loss_kl = torch.tensor(loss_dict['KL']).cpu().numpy()

In [ ]:
n_epochs = train_configs.n_epochs
beta = model_configs.beta

step = 10

plt.figure(figsize=(6, 4))
plt.plot(np.arange(n_epochs)[::step], loss_tot[::step], '.--',  label = 'Total loss')
plt.plot(np.arange(n_epochs)[::step], loss_nll[::step], '.--', color='orange', label = r'$\Vert x - \hat{x} \Vert^2$')
plt.plot(np.arange(n_epochs)[::step], loss_kl[::step], '.--', color='green', label = r'$D_{\text{KL}}(q(z \mid x) || p(z))$')

plt.title('Training logs')
plt.legend()
plt.show()


In [ ]:
torch.save(model.state_dict(), 'results/bvae_2d_lowres_std_norm.pt')

#### Benchmark (held-out log-likelihood)

In [ ]:
model_1d_configs = configs.set_model_configs(
    drop_rate = 0.2,
    beta = 0.33,
    pz_std = 1,
    c_base = 128,
    layer_mults = [8, 4],
    ydim = 128, 
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

model_1d = bvae.BetaVAE(model_1d_configs)
model_1d.load_state_dict(torch.load('results/bvae_1d_lowres_std_norm.pt'))

model_2d_configs = configs.set_model_configs(
    drop_rate = 0.2,
    beta = 0.33,
    pz_std = 1,
    ydim = 128,
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

model_2d = bvae.BetaVAE2D(model_2d_configs)
model_2d.load_state_dict(torch.load('results/bvae_2d_lowres_std_norm.pt'))

dataset = CyIFDataset(data_path='../data/cycif/zonation_lowres_test/')
dataloader = DataLoader(dataset, batch_size=model_1d_configs.batch_size, shuffle=False)

In [ ]:
imgs, x_preds_1d, _ = model_eval.predict(model_1d, 
                                         dataloader = dataloader,
                                         device = model_1d_configs.device, 
                                         batch_size = 1)

_, x_preds_2d, _ = model_eval.predict(model_2d,
                                      dataloader = dataloader,
                                      device = model_2d_configs.device,
                                      batch_size = 1)

In [ ]:
MSEs_1d = [((x_true - x_pred)**2).mean()
           for (x_true, x_pred) in zip(imgs, x_preds_1d)]
MSEs_2d = [((x_true - x_pred)**2).mean()
           for (x_true, x_pred) in zip(imgs, x_preds_2d)]

plot_df = pd.DataFrame({
    'MSE': MSEs_1d + MSEs_2d,
    'Label': ['VAE (FC)']*len(imgs) + ['VAE (Conv2D)']*len(imgs)
})

sns.boxplot(x='Label', y='MSE', data=plot_df, hue='Label')
plt.title('Held-out avg. per-pixel MSE')
plt.show()

#### Predictions & Assessment

In [ ]:
model_1d_configs = configs.set_model_configs(
    drop_rate = 0.1,
    beta = 1,
    pz_std = 1,
    c_base = 128,
    layer_mults = [8, 4],
    ydim = 128, 
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

model_2d_configs = configs.set_model_configs(
    drop_rate = 0.1,
    beta = 1,
    ydim = 128,
    xdim = 128,
    batch_size=1,
    device = torch.device('cpu')
)

In [ ]:
dataset = IMSDataset(
    data_path='../data/cycif/zonation_lowres/',
    norm_stats=(constants.CYIF_MEAN, constants.CYIF_STD)
)

do_2d = True
if do_2d:
    model_configs = model_2d_configs
    model = bvae.BetaVAE2D(model_configs)
    dataloader = DataLoader(dataset, batch_size=model_configs.batch_size, shuffle=False)
    model.load_state_dict(torch.load('results/bvae_2d_lowres_std_norm.pt'))
else:
    model_configs = model_1d_configs
    model = bvae.BetaVAE(model_configs)
    dataloader = DataLoader(dataset, batch_size=model_configs.batch_size, shuffle=False)
    model.load_state_dict(torch.load('results/bvae_1d_lowres_std_norm.pt'))

In [ ]:
imgs, x_preds, latents, qz_preds = model_eval.predict(model, 
                                                      dataloader=dataloader,
                                                      device=model_configs.device, 
                                                      batch_size=1)

---